# Modèle de Prévision des Ruptures de Stock

---

## Objectif de ce notebook

Estimer le **nombre de jours restants avant rupture de stock** pour le Gasoil au Dépôt Central Lomé.  
Ce modèle alimente les **compteurs colorés** du dashboard (vert / orange / rouge) indiquant  
l'urgence de réapprovisionner chaque dépôt.

**Couple dépôt/produit :** Gasoil (PRD003) + Dépôt Central Lomé (D001)  
**Algorithmes comparés :** XGBoost vs Random Forest (régression)  
**Variable cible :** `jours_couverture` — calculée à partir du stock et de la consommation moyenne  
**Métriques :** MAE, RMSE

---

## 0. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Dossiers racine pour les figures et modèles
FIGURES_DIR = os.path.join('..', 'figures')
MODELS_DIR = os.path.join('..', 'models')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

_original_savefig = plt.savefig
def savefig_root(path, *args, **kwargs):
    if isinstance(path, str) and path.startswith('figures/'):
        path = os.path.join('..', path)
    return _original_savefig(path, *args, **kwargs)

plt.savefig = savefig_root

print('Imports OK ✅')

## 1. Chargement des Données

In [ ]:
df_full = pd.read_csv('../data/stocks_journaliers.csv')
df_full['date'] = pd.to_datetime(df_full['date'])

df = df_full[
    (df_full['produit_id'] == 'PRD003') &
    (df_full['depot_id'] == 'D001')
].copy().sort_values('date').reset_index(drop=True)

print(f"Série : Gasoil — Dépôt Central Lomé")
print(f"Lignes  : {len(df):,}")
print(f"Période : {df['date'].min().date()} → {df['date'].max().date()}")
df[['date', 'stock_fin_jour', 'sorties', 'taux_remplissage_pct']].head()

## 2. Calcul de la Variable Cible — `jours_couverture`

La variable cible n'existe pas directement dans le dataset — elle doit être **calculée**.  

**Formule retenue :**
$$\text{jours\_couverture} = \frac{\text{stock\_fin\_jour}}{\text{consommation\_moyenne\_7j}}$$

**Justification du dénominateur (moyenne 7 jours) :**  
- La consommation du jour seul est trop volatile (bruit journalier important, std=210 litres)  
- La moyenne sur 30 jours est trop lissée et réagit trop lentement aux changements récents  
- La moyenne sur **7 jours** est le bon compromis : elle capture la tendance récente  
  tout en filtrant le bruit quotidien, et correspond au cycle hebdomadaire observé à l'EDA

In [ ]:
# Calcul de la consommation moyenne sur 7 jours glissants
df['conso_ma7'] = df['sorties'].rolling(window=7, min_periods=1).mean()

# Calcul des jours de couverture
# On clippe à 365 pour éviter des valeurs aberrantes (stock très élevé + conso très faible)
df['jours_couverture'] = (df['stock_fin_jour'] / df['conso_ma7'].replace(0, np.nan)).clip(0, 365)

print("=== Statistiques : jours_couverture ===")
print(df['jours_couverture'].describe().round(2))
print()
print("=== Seuils d'alerte ===")
print(f"  < 7  jours (CRITIQUE) : {(df['jours_couverture'] < 7).sum():>4} obs ({(df['jours_couverture'] < 7).mean()*100:.1f}%)")
print(f"  < 14 jours (ÉLEVÉ)    : {(df['jours_couverture'] < 14).sum():>4} obs ({(df['jours_couverture'] < 14).mean()*100:.1f}%)")
print(f"  < 30 jours (MODÉRÉ)   : {(df['jours_couverture'] < 30).sum():>4} obs ({(df['jours_couverture'] < 30).mean()*100:.1f}%)")
print(f"  >= 30 jours (OK)      : {(df['jours_couverture'] >= 30).sum():>4} obs ({(df['jours_couverture'] >= 30).mean()*100:.1f}%)")

In [ ]:
# Visualisation de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Évolution temporelle
axes[0].plot(df['date'], df['jours_couverture'], color='steelblue', linewidth=0.7)
axes[0].axhline(30, color='orange', linestyle='--', linewidth=1.2, label='Seuil modéré (30j)')
axes[0].axhline(14, color='red', linestyle='--', linewidth=1.2, label='Seuil élevé (14j)')
axes[0].set_title('Évolution des jours de couverture (2015–2024)')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Jours de couverture')
axes[0].legend()

# Distribution
axes[1].hist(df['jours_couverture'], bins=50, color='steelblue', edgecolor='white')
axes[1].axvline(df['jours_couverture'].mean(), color='darkorange',
                linestyle='--', linewidth=1.5,
                label=f"Moyenne : {df['jours_couverture'].mean():.0f}j")
axes[1].set_title('Distribution des jours de couverture')
axes[1].set_xlabel('Jours de couverture')
axes[1].set_ylabel('Fréquence')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/01_variable_cible_couverture.png', dpi=150)
plt.show()

**📝 Observations :**

> La variable cible `jours_couverture` présente une **moyenne de 131,9 jours** avec un écart-type de 19,1 jours — les stocks sont globalement bien gérés sur ce dépôt. La distribution est approximativement normale, centrée autour de 134 jours. Aucune observation ne passe sous les seuils critiques de 7 ou 14 jours sur ce couple dépôt/produit, ce qui confirme la bonne gestion du Dépôt Central Lomé.
>
> La variable `stock_fin_jour` est fortement corrélée aux jours de couverture (r=0.809) — logique puisqu'elle est au numérateur de la formule. Les sorties sont modérément corrélées négativement (r=-0.217) — une consommation plus élevée réduit la couverture.

## 3. Feature Engineering

On enrichit le dataset avec des variables temporelles, des lags et des moyennes glissantes  
pour donner aux modèles une vision à la fois locale (court terme) et globale (long terme).

In [ ]:
# Variables temporelles
df['jour_semaine'] = df['date'].dt.dayofweek
df['mois']         = df['date'].dt.month
df['trimestre']    = df['date'].dt.quarter
df['is_weekend']   = (df['date'].dt.dayofweek >= 5).astype(int)

# Moyennes glissantes sur le stock et les sorties
df['stock_ma7']    = df['stock_fin_jour'].rolling(7,  min_periods=1).mean()
df['stock_ma14']   = df['stock_fin_jour'].rolling(14, min_periods=1).mean()
df['sorties_ma7']  = df['sorties'].rolling(7,  min_periods=1).mean()
df['sorties_ma14'] = df['sorties'].rolling(14, min_periods=1).mean()
df['sorties_ma30'] = df['sorties'].rolling(30, min_periods=1).mean()

# Lag features sur les jours de couverture
df['couverture_lag1']  = df['jours_couverture'].shift(1)
df['couverture_lag7']  = df['jours_couverture'].shift(7)
df['couverture_lag14'] = df['jours_couverture'].shift(14)

# Tendance du stock : est-ce que le stock monte ou descend ?
df['tendance_stock'] = df['stock_fin_jour'] - df['stock_fin_jour'].shift(7)

# Supprimer les NaN créés par les lags
df = df.dropna().reset_index(drop=True)

# Définition des features retenues
FEATURES = [
    'stock_fin_jour', 'taux_remplissage_pct',
    'sorties', 'entrees',
    'stock_ma7', 'stock_ma14',
    'sorties_ma7', 'sorties_ma14', 'sorties_ma30',
    'couverture_lag1', 'couverture_lag7', 'couverture_lag14',
    'tendance_stock',
    'jour_semaine', 'mois', 'trimestre', 'is_weekend',
    'prix_wti_usd_baril'
]
TARGET = 'jours_couverture'

print(f"Dataset final : {len(df):,} lignes")
print(f"Features      : {len(FEATURES)} variables")
print(f"\nListe des features :")
for f in FEATURES:
    print(f"  {f}")

## 4. Séparation Train / Test

Même séparation chronologique que le notebook précédent : **2015–2022 / 2023–2024**.

In [ ]:
CUTOFF = '2023-01-01'

train = df[df['date'] < CUTOFF].copy()
test  = df[df['date'] >= CUTOFF].copy()

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"Train : {len(train):,} observations")
print(f"Test  : {len(test):,}  observations")
print(f"Features : {len(FEATURES)}")

## 5. Modèle 1 — XGBoost

XGBoost (eXtreme Gradient Boosting) est un algorithme d'ensemble basé sur le boosting.  
Il construit des arbres de décision séquentiellement, chaque arbre corrigeant les erreurs  
du précédent. Il est particulièrement efficace sur des données tabulaires avec de nombreuses features.

In [ ]:
# Configuration et entraînement XGBoost
model_xgb = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
model_xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
print("XGBoost entraîné ✅")

# Prévisions
y_pred_xgb = model_xgb.predict(X_test)

mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
print(f"\n=== Métriques XGBoost ===")
print(f"  MAE  : {mae_xgb:.2f} jours")
print(f"  RMSE : {rmse_xgb:.2f} jours")

In [ ]:
# Prévisions vs Réel — XGBoost
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test['date'], y_test.values,
        color='steelblue', linewidth=0.8, label='Réel')
ax.plot(test['date'], y_pred_xgb,
        color='darkorange', linewidth=1.0,
        linestyle='--', label='XGBoost')
ax.axhline(30, color='red', linestyle=':', linewidth=1.0, label='Seuil alerte (30j)')
ax.set_title('XGBoost — Jours de couverture prévus vs réels (2023–2024)')
ax.set_xlabel('Date')
ax.set_ylabel('Jours de couverture')
ax.legend()
plt.tight_layout()
plt.savefig('figures/02_xgb_previsions.png', dpi=150)
plt.show()

In [ ]:
# Importance des variables — XGBoost
importance_xgb = pd.Series(
    model_xgb.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
importance_xgb.tail(15).plot(
    kind='barh', ax=ax, color='darkorange', edgecolor='white'
)
ax.set_title('XGBoost — Importance des 15 variables les plus influentes')
ax.set_xlabel('Importance (gain)')
plt.tight_layout()
plt.savefig('figures/03_xgb_importance.png', dpi=150)
plt.show()

print("Top 5 variables les plus importantes :")
for feat, imp in importance_xgb.tail(5).iloc[::-1].items():
    print(f"  {feat:<30} : {imp:.4f}")

## 6. Modèle 2 — Random Forest

Random Forest construit un grand nombre d'arbres de décision en parallèle (bagging),  
chacun entraîné sur un sous-échantillon aléatoire des données et des features.  
La prédiction finale est la moyenne des prédictions de tous les arbres.  
Il est plus robuste au surapprentissage que XGBoost sur des datasets de taille moyenne.

In [ ]:
# Configuration et entraînement Random Forest
model_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
model_rf.fit(X_train, y_train)
print("Random Forest entraîné ✅")

# Prévisions
y_pred_rf = model_rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
print(f"\n=== Métriques Random Forest ===")
print(f"  MAE  : {mae_rf:.2f} jours")
print(f"  RMSE : {rmse_rf:.2f} jours")

In [ ]:
# Importance des variables — Random Forest
importance_rf = pd.Series(
    model_rf.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
importance_rf.tail(15).plot(
    kind='barh', ax=ax, color='steelblue', edgecolor='white'
)
ax.set_title('Random Forest — Importance des 15 variables les plus influentes')
ax.set_xlabel('Importance (gini)')
plt.tight_layout()
plt.savefig('figures/04_rf_importance.png', dpi=150)
plt.show()

print("Top 5 variables les plus importantes :")
for feat, imp in importance_rf.tail(5).iloc[::-1].items():
    print(f"  {feat:<30} : {imp:.4f}")

## 7. Comparaison et Sélection du Meilleur Modèle

In [ ]:
# Tableau comparatif
resultats = pd.DataFrame({
    'Modèle' : ['XGBoost', 'Random Forest'],
    'MAE (jours)'  : [mae_xgb,  mae_rf],
    'RMSE (jours)' : [rmse_xgb, rmse_rf]
})
print(resultats.to_string(index=False))

In [ ]:
# Graphique comparatif prévisions vs réel
n = 120
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test['date'].values[:n], y_test.values[:n],
        color='steelblue', linewidth=1.0, label='Réel', zorder=3)
ax.plot(test['date'].values[:n], y_pred_xgb[:n],
        color='darkorange', linewidth=1.1,
        linestyle='--', label='XGBoost')
ax.plot(test['date'].values[:n], y_pred_rf[:n],
        color='green', linewidth=1.1,
        linestyle=':', label='Random Forest')
ax.axhline(30, color='red', linestyle=':', linewidth=1.0, label='Seuil alerte 30j')
ax.set_title('Comparaison XGBoost vs Random Forest — 120 premiers jours du test')
ax.set_xlabel('Date')
ax.set_ylabel('Jours de couverture prévus')
ax.legend()
plt.tight_layout()
plt.savefig('figures/05_comparaison_modeles.png', dpi=150)
plt.show()

In [ ]:
# Comparaison importance des variables entre les deux modèles
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

importance_xgb.tail(10).plot(
    kind='barh', ax=axes[0], color='darkorange', edgecolor='white'
)
axes[0].set_title('XGBoost — Top 10 variables')

importance_rf.tail(10).plot(
    kind='barh', ax=axes[1], color='steelblue', edgecolor='white'
)
axes[1].set_title('Random Forest — Top 10 variables')

plt.suptitle('Comparaison de l\'importance des variables', fontsize=13)
plt.tight_layout()
plt.savefig('figures/06_comparaison_importances.png', dpi=150)
plt.show()

In [ ]:
# Barplot des métriques
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
metriques = ['MAE (jours)', 'RMSE (jours)']
valeurs   = [[mae_xgb, mae_rf], [rmse_xgb, rmse_rf]]

for i, (metrique, vals) in enumerate(zip(metriques, valeurs)):
    bars = axes[i].bar(
        ['XGBoost', 'Random Forest'], vals,
        color=['darkorange', 'steelblue'], edgecolor='white', width=0.5
    )
    for bar, val in zip(bars, vals):
        axes[i].text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(vals)*0.01,
            f'{val:.2f}j', ha='center', fontsize=11
        )
    axes[i].set_title(metrique)
    axes[i].set_ylabel(metrique)

fig.suptitle('Comparaison MAE / RMSE — XGBoost vs Random Forest', fontsize=13)
plt.tight_layout()
plt.savefig('figures/07_barplot_metriques.png', dpi=150)
plt.show()

**📝 Sélection et Justification :**

> Les deux modèles sont comparés sur le MAE et le RMSE — le MAPE n'est pas utilisé ici car la variable cible (jours de couverture) peut prendre des valeurs très élevées (>100 jours), ce qui rendrait le MAPE peu informatif.
>
> **Interprétation métier des erreurs :** avec un délai fournisseur moyen de **8,5 jours** (observé dans `bons_commande`), une erreur de prédiction de X jours a les conséquences suivantes :
> - Si le modèle **sous-estime** de X jours (prédit 20j alors que c'est 28j) → la commande est déclenchée trop tôt → surstock temporaire, coût financier faible
> - Si le modèle **surestime** de X jours (prédit 20j alors que c'est 12j) → la commande est déclenchée trop tard → risque de rupture si X > délai fournisseur (8,5j)
>
> Dans ce contexte, **sous-estimer est préférable à surestimer** — le système d'alertes appliquera donc un **coefficient de sécurité de 1.2** sur les seuils d'alerte pour compenser l'incertitude du modèle.
>
> **Comparaison des importances :** les deux modèles s'accordent sur les variables les plus influentes — `couverture_lag1`, `stock_fin_jour` et `stock_ma7` dominent dans les deux cas. Cela confirme que les hypothèses formulées lors de l'EDA (stock_fin_jour et taux_remplissage très corrélés à la couverture) sont bien captées par les modèles. Le `prix_wti_usd_baril` a peu d'importance — cohérent avec sa faible corrélation (r=-0.021) observée lors de l'exploration.

## 8. Sauvegarde du Modèle

In [ ]:
# Sauvegarder le meilleur modèle
# Remplacer MODEL_FINAL par le modèle ayant le MAE le plus bas
MODEL_FINAL = model_xgb if mae_xgb <= mae_rf else model_rf
NOM_FINAL   = 'XGBoost' if mae_xgb <= mae_rf else 'Random Forest'

with open('../models/model_ruptures.pkl', 'wb') as f:
    pickle.dump(MODEL_FINAL, f)

print(f"Modèle retenu  : {NOM_FINAL}")
print(f"Fichier créé   : ../models/model_ruptures.pkl ✅")
print()
print("=== Documentation pour l'intégration API ===")
print()
print("Format des données d'entrée :")
print("  DataFrame avec les colonnes suivantes (dans cet ordre) :")
for f in FEATURES:
    print(f"    - {f}")
print()
print("Format de la sortie :")
print("  Array numpy de shape (n,) — nombre de jours de couverture prédit")
print()
print("Exemple d'appel dans l'API :")
print("  import pickle")
print("  with open('../models/model_ruptures.pkl', 'rb') as f:")
print("      model = pickle.load(f)")
print("  predictions = model.predict(X_new)")

## 9. Conclusions

---

### 🔍 Résultats clés

> 1. La variable cible `jours_couverture` a été **calculée** à partir du stock et de la consommation moyenne sur 7 jours — ce choix est justifié par le cycle hebdomadaire observé à l'EDA et la volatilité journalière élevée (std=210 litres).
> 2. **18 features** ont été créées, dominées par les lags de couverture et les moyennes glissantes du stock.
> 3. Les deux modèles produisent des résultats proches — les variables les plus importantes sont cohérentes entre XGBoost et Random Forest.
> 4. **Interprétation clé :** une erreur de prédiction supérieure au délai fournisseur (8,5j) peut conduire à une rupture → le système applique un coefficient de sécurité de -15% sur les prévisions.

---

### ✅ Décisions

> - **Modèle retenu** : celui avec le MAE le plus bas (évalué à l'exécution)
> - **Fichier sauvegardé** : `models/model_ruptures.pkl`
> - **Seuils d'alerte dashboard** : < 30j (orange), < 14j (rouge), < 7j (critique)
> - **Coefficient de sécurité** : ×0.85 appliqué sur les prévisions en production

---

### 📁 Figures produites

| Fichier | Description |
|---|---|
| `01_variable_cible_couverture.png` | Évolution et distribution de la variable cible |
| `02_xgb_previsions.png` | Prévisions XGBoost vs réel |
| `03_xgb_importance.png` | Importance des variables XGBoost |
| `04_rf_importance.png` | Importance des variables Random Forest |
| `05_comparaison_modeles.png` | Superposition XGBoost vs RF vs réel |
| `06_comparaison_importances.png` | Comparaison des importances côte à côte |
| `07_barplot_metriques.png` | Comparaison MAE / RMSE |